# 🏦 Rule-Based Synthetic Data with Business Constraints

In this notebook we will build a rule-based synthetic data script that enforces key business constraints such as valid ranges and consistent categorical values, then validate each rule.

We have the following scenario: 

You've built a banking dashboard application and need realistic transactional data to test it — but you have no real customer data available.
1. Define the data requirements as a structured checklist
2. Translate the requirements into a schema
3. Generate 1,000 synthetic banking transactions
4. Validate the generated dataset

## 1. Define Requirements & import package

| # | Question | Answer (our banking transactions) |
|---|---|---|
| ✅ | **Columns** | `transaction_id`, `account_id`, `transaction_date`, `transaction_type`, `merchant`, `amount`, `country` |
| ✅ | **Valid ranges** | Amounts: €1 – €5,000 · Dates: 2023-01-01 to 2024-12-31 |
| ✅ | **Constraints** | Country code must be `DE` (Germany) · Amount > 0 and ≤ €5,000 · No null values · `transaction_id` is unique |
| ✅ | **Expected patterns** | 60% debits / 40% credits |

In [1]:
# Required packages
from ydata.metadata import Metadata
from ydata.metadata.builder import MetadataConfigurationBuilder
from ydata.synthesizers import FakerSynthesizer

## 2. Define the Schema

We translate each requirement directly into a schema dictionary.
Each key is a column name; the value describes its type, format, and constraints.

In [2]:
from datetime import datetime

schema = {
    # --- Identifiers ---
    "transaction_id": {
        "datatype":  "string",
        "vartype":   "string",
        "regex":     "TXN-[0-9]{8}",          # e.g. TXN-00482931
    },
    "account_id": {
        "datatype":  "string",
        "vartype":   "string",
        "regex":     "ACC-[0-9]{6}",           # e.g. ACC-004829
    },

    # --- When ---
    "transaction_date": {
        "datatype":  "date",
        "vartype":   "date",
        "min":       datetime(2023, 1, 1),
        "max":       datetime(2024, 12, 31),
        "format":    "%Y-%m-%d",
    },

    # --- What kind ---
    "transaction_type": {
        "datatype":    "categorical",
        "vartype":     "string",
        "categories": {"debit": 60, "credit": 40},  # 60% debits, 40% credits
    },

    # --- Where ---
    "merchant": {
        "datatype":  "string",
        "vartype":   "string",
        "characteristic": "name",               # realistic merchant names via Faker
    },

    # --- How much ---
    "amount": {
        "datatype":  "numerical",
        "vartype":   "float",
        "min":       1,
        "max":       5_000,                     # transaction amounts: €1 – €5,000
    },

    # --- Country ---
    "country_code": {
        "datatype":    "categorical",
        "vartype":     "string",
        "categories": {"DE": 40, "US": 30, "GB": 20, "FR": 10},             # Germany only (ISO 3166-1 alpha-2)
    },
}

print(f"Schema defined: {len(schema)} columns ✓")
for col, cfg in schema.items():
    print(f"  • {col:<20} {cfg['datatype']}")

Schema defined: 7 columns ✓
  • transaction_id       string
  • account_id           string
  • transaction_date     date
  • transaction_type     categorical
  • merchant             string
  • amount               numerical
  • country_code         categorical


## 3. Build Metadata & Initialise the Synthesizer

We pass the schema dictionary to `MetadataConfigurationBuilder`, which creates a `Metadata` object — a reusable blueprint that describes the dataset structure. We can save it and regenerate identical-structure datasets at any time.

In [4]:
from ydata.metadata import Metadata
from ydata.metadata.builder import MetadataConfigurationBuilder
from ydata.synthesizers import FakerSynthesizer

# Build metadata from the schema dictionary
builder = MetadataConfigurationBuilder(schema)
metadata = Metadata(configuration_builder=builder)

print("Metadata object created ✓")
print(metadata)

Metadata object created ✓
Metadata Summary 
 
Dataset type: TABULAR
Dataset attributes: 
Number of columns: 7
Number of rows: 0
Duplicate rows: 0
Target column: 

Column detail: 
             Column    Data type Variable type Characteristics
0    transaction_id       string        string                
1        account_id       string        string                
2  transaction_date         date          date                
3  transaction_type  categorical        string                
4          merchant       string        string            name
5            amount    numerical         float                
6      country_code  categorical        string                
Warnings: 
No warning!



## 4. Fit & Generate

When using addresses, names, and other properties that depend on a language, you can set the `FakerSynthesizer` language by using the property locale.

In [5]:
# Instantiate with English locale
synth = FakerSynthesizer(locale="en")

# Fit on the metadata (learns the schema structure)
synth.fit(metadata)

# Generate 1,000 transactions
# You can request as many rows as you need
sample = synth.sample(sample_size=1000)
df = sample.to_pandas()

print(f"Generated: {df.shape[0]:,} rows × {df.shape[1]} columns ✓")
df.head()

Generated: 1,000 rows × 7 columns ✓


,transaction_id,account_id,transaction_date,transaction_type,merchant,amount,country_code
0,TXN-42248305,ACC-126541,2023-06-28,credit,Antonio Taylor,3531.659624,US
1,TXN-67808756,ACC-310481,2024-05-11,debit,Lawrence Thomas,1231.163136,GB
2,TXN-91386180,ACC-863257,2024-09-29,debit,Teresa Palmer,3325.910610,DE
3,TXN-34360027,ACC-401751,2023-10-14,credit,Aaron Brown,4425.875675,DE
4,TXN-85825729,ACC-950736,2024-11-01,debit,Jacob Olsen,381.605149,US


In [10]:
# Derive the country name from the country code
CODE_TO_COUNTRY = {
    "DE": "Germany",
    "US": "USA",
    "GB": "UK",
    "FR": "France",
}

df["country"] = df["country_code"].map(CODE_TO_COUNTRY)

print("Country name column derived ✓")
df[["country_code", "country"]].value_counts().sort_index()

Country name column derived ✓


country_code  country
DE            Germany    400
FR            France     100
GB            UK         200
US            USA        300
Name: count, dtype: int64

## 5. Define Constraints with the Constraint Engine

We use ydata-sdk's ConstraintEngine to validate that the generated dataset follows the user defined rules.

| Constraint | Rule |
|---|---|
| `NotNull` | No column may contain missing values |
| `Unique` | Every `transaction_id` must be unique |
| `MaxBetween(0, 5000)` | No amount may exceed €5,000 |
| `CustomConstraint` | Country code must be mapped to the column Country

`GreaterThan` and `CustomConstraint` are **row-level**: they tag non-compliant rows and can filter them out.
`MaxBetween` is a **column-level** summary check: it verifies the column's maximum falls within a bound.

In [9]:
from ydata.constraints import (
    ConstraintEngine,
    NotNull,
    Unique,
    GreaterThan,
    MaxBetween,
    CustomConstraint,
)
from ydata.dataset import Dataset

# Define one constraint per business rule
engine = ConstraintEngine([
    
    # Not Null validated the presence of missing values
    NotNull(
        columns=list(df.columns),
        name="no_missing_values",
    ),

    # Unique, checks if all the values in a defined columns are unique
    Unique(
        columns=["transaction_id"],
        name="unique_transaction_id",
    ),

    # MaxBetween validates if the values for a numerical columns are within the expected ranges
    MaxBetween(
        lower_bound=0,
        upper_bound=5_000,
        columns=["amount"],
        name="amount_ceiling",
    ),

    # CustomerConstraint allows you to define constraint as function
    # For country_code it was implemented a lambda function with a mapping
    CustomConstraint(
    lambda row: CODE_TO_COUNTRY.get(row["country_code"]) == row["country"],
    columns=["country_code", "country"],
    name="country_code_consistency",
    axis="row",
    ),
])

# Wrap the DataFrame in a Dataset so the engine can process it
dataset = Dataset(df)

# Run all constraints
engine.validate(dataset)

# Show a summary table: one row per constraint
engine.summary()

{'rows_violation_count': np.int64(0),
 'rows_violation_ratio': 0.0,
 'violation_per_constraint': {'no_missing_values': {'rows_violation_count': np.int64(0),
   'rows_violation_ratio': 0.0,
   'validation_time': 0.0},
  'unique_transaction_id': {'rows_violation_count': np.int64(0),
   'rows_violation_ratio': 0.0,
   'validation_time': 0.0},
  'amount_ceiling': {'column_violation_count': np.int64(0),
   'column_violation_ratio': 0.0,
   'violated_columns': [],
   'validation_time': 0.0},
  'country_code_consistency': {'rows_violation_count': np.int64(0),
   'rows_violation_ratio': 0.0,
   'validation_time': 0.0}}}

## ✅ Summary

In this notebook you:

1. **Defined requirements** as a structured four-question checklist before writing any code
2. **Translated them into a schema** dictionary covering identifiers, dates, categories, and numeric ranges.
3. **Generated 1,000 synthetic banking transactions** with `FakerSynthesizer`
5. **Declared business rules as constraint objects** for ease of validation and re-use.
6. **Validated the generated datasets**